# Caduceus (SSM) with Real Genomic DNA

Tests SSM scaling behavior on **real genomic sequences** (not synthetic DNA).

## Purpose

- **Synthetic DNA (other notebook)**: Tests pure architectural properties
- **Real DNA (this notebook)**: Ecological validity for SSMs

Uses human chromosome 22 for diverse genomic regions.

## Setup Instructions

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (Runtime > Change runtime type > GPU)
3. Run cells in order

---

In [ ]:
# Install Dependencies
#
# Strategy: Build mamba-ssm from source to match Colab's CUDA/PyTorch.
# Prebuilt cu11 wheels are ABI-incompatible with Colab's CUDA 12 PyTorch.
# --no-build-isolation ensures it uses the already-installed torch/CUDA.
#
# This takes ~10-15 min on first run but only happens once per session.
# Go get coffee. Once built, everything loads natively with full pretrained
# weights -- no shim, no missing keys, no NaN.

print("Installing core dependencies...")
!pip install -q torch transformers shesha-geometry matplotlib seaborn pandas einops

# Clean any stale cu11 binaries that cause ABI mismatch
print("\nCleaning stale mamba installs...")
!pip uninstall -y mamba-ssm selective-scan-cuda causal-conv1d 2>/dev/null; true

print("\nBuilding causal-conv1d from source...")
!pip install causal-conv1d --no-build-isolation

print("\nBuilding mamba-ssm from source (this takes ~10-15 min, one-time cost)...")
!pip install mamba-ssm --no-build-isolation

# Verify
import torch
print(f"\ntorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

import mamba_ssm
print(f"mamba-ssm {mamba_ssm.__version__} -- native CUDA kernels OK!")
from mamba_ssm.modules.mamba_simple import Mamba
print("Mamba class imported successfully. No shim needed.")

print("\nDone! Native mamba-ssm is ready.")


In [ ]:
# Configuration

PHASE = 'full'  # 'quick' or 'full'

import os
import sys
sys.path.insert(0, '.')
OUTPUT_BASE = './results/caduceus_realdna_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

# Use caduceus-PS (Parameter Sharing) models -- these are RC-equivariant.
# PS models have rcps=True natively: the RCPSWrapper flips inputs for the
# reverse branch and ties weights, giving true Reverse Complement equivariance.
# PH models (caduceus-ph_*) are bidirectional but NOT RC-equivariant.
CONFIG = {
    'quick': {
        'n_sequences': 500,
        'seq_length': 1000,
        'models': [
            'kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3',   # 471k
            'kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',        # 7.73M
        ],
        'batch_size': 8,
        'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10],
    },
    'full': {
        'n_sequences': 10000,
        'seq_length': 1000,
        'models': [
            'kuleshov-group/caduceus-ps_seqlen-1k_d_model-118_n_layer-4_lr-8e-3',   # 471k
            'kuleshov-group/caduceus-ps_seqlen-1k_d_model-256_n_layer-4_lr-8e-3',   # 1.93M
            'kuleshov-group/caduceus-ps_seqlen-131k_d_model-256_n_layer-16',        # 7.73M
        ],
        'batch_size': 4,
        'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10],
    },
}

config = CONFIG[PHASE]
print(f"Phase: {PHASE.upper()}")
print(f"Architecture: Caduceus (SSM)")
print(f"Data: Real human genomic DNA (chr22)")
print(f"Sequences: {config['n_sequences']}")
print(f"Models: {len(config['models'])}")

In [ ]:
# Load Real Genomic DNA
#
# Downloads human chr22 (~50MB) and randomly samples regions.
# Oversamples by 50% buffer to account for N-rich regions being filtered.

import urllib.request
import gzip
import os
import numpy as np

CHR22_URL = 'https://hgdownload.soe.ucsc.edu/goldenPath/hg38/chromosomes/chr22.fa.gz'
VALID_BASES = set('ACGT')


def download_and_sample_genomic_dna(n_sequences, seq_length, seed=320):
    cache_file = f'{CACHE_DIR}/chr22_sample_{n_sequences}_{seq_length}.txt'

    if os.path.exists(cache_file):
        print(f"Loading cached genomic sequences: {cache_file}")
        with open(cache_file) as f:
            sequences = [line.strip() for line in f if line.strip()]
        print(f"Loaded {len(sequences)} cached sequences")
        return sequences

    print("Downloading human chr22 (~50MB, first run only)...")

    try:
        with urllib.request.urlopen(CHR22_URL) as response:
            with gzip.GzipFile(fileobj=response) as f:
                lines = f.read().decode('utf-8').split('\n')
                chr22_seq = ''.join(line.strip() for line in lines[1:] if line.strip())

        print(f"Downloaded chr22 ({len(chr22_seq):,} bp)")

        # Oversample with buffer to guarantee enough clean sequences
        buffer_factor = 1.5
        max_attempts = int(n_sequences * buffer_factor)
        print(f"Sampling regions (target={n_sequences}, buffer={max_attempts})...")

        rng = np.random.default_rng(seed)
        sequences = []

        for attempt in range(max_attempts):
            start = rng.integers(0, len(chr22_seq) - seq_length)
            seq = chr22_seq[start:start + seq_length].upper()

            # Filter: less than 10% unknown bases
            n_count = sum(1 for c in seq if c not in VALID_BASES)
            if n_count < seq_length * 0.1:
                # Replace remaining non-ACGT with random bases
                seq_clean = ''.join(
                    c if c in VALID_BASES else rng.choice(['A', 'C', 'G', 'T'])
                    for c in seq
                )
                sequences.append(seq_clean)

            if len(sequences) >= n_sequences:
                break

        sequences = sequences[:n_sequences]
        print(f"Sampled {len(sequences)} clean sequences")

        if len(sequences) < n_sequences:
            print(f"WARNING: Only got {len(sequences)}/{n_sequences}")

        # Cache
        os.makedirs(CACHE_DIR, exist_ok=True)
        with open(cache_file, 'w') as f:
            f.write('\n'.join(sequences))
        print(f"Cached to {cache_file}")

        return sequences

    except Exception as e:
        print(f"Error: {e}")
        print("Fallback: generating synthetic DNA...")
        rng = np.random.default_rng(seed)
        return [
            ''.join(rng.choice(['A', 'C', 'G', 'T'], size=seq_length))
            for _ in range(n_sequences)
        ]


sequences = download_and_sample_genomic_dna(
    config['n_sequences'],
    config['seq_length'],
    seed=320,
)

print(f"\nSequence stats:")
print(f"  Count: {len(sequences)}")
print(f"  Length: {len(sequences[0])}")
print(f"  Sample: {sequences[0][:60]}...")

# Validate base composition
total_bases = sum(len(s) for s in sequences)
print(f"\nBase composition:")
for base in 'ACGT':
    count = sum(s.count(base) for s in sequences)
    print(f"  {base}: {count/total_bases*100:.1f}%")
non_acgt = sum(1 for s in sequences for c in s if c not in VALID_BASES)
print(f"  Non-ACGT bases: {non_acgt}")

In [ ]:
# DNA Perturbation Suite

from dataclasses import dataclass, field

DNA_BASES = ['A', 'C', 'G', 'T']
COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

@dataclass
class PerturbedSet:
    name: str
    category: str
    sequences: list
    params: dict = field(default_factory=dict)
    description: str = ''


def mutate_dna(sequence, mutation_rate, rng):
    """SNP mutations. Only mutates valid ACGT bases."""
    seq = list(sequence)
    valid_positions = [i for i, c in enumerate(seq) if c in DNA_BASES]
    n_mutations = max(1, int(len(valid_positions) * mutation_rate))
    positions = rng.choice(valid_positions, size=min(n_mutations, len(valid_positions)), replace=False)
    for pos in positions:
        original = seq[pos]
        alternatives = [b for b in DNA_BASES if b != original]
        seq[pos] = rng.choice(alternatives)
    return ''.join(seq)


def reverse_complement(sequence):
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(sequence))


class DNAPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_rc=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_rc = include_rc

    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"snp_{int(rate * 100)}pct"
            perturbed = [mutate_dna(s, rate, self.rng) for s in sequences]
            results[name] = PerturbedSet(
                name=name, category='snp', sequences=perturbed,
                params={'mutation_rate': rate},
                description=f'SNP mutations at {rate*100:.0f}% rate',
            )
        if self.include_rc:
            results['reverse_complement'] = PerturbedSet(
                name='reverse_complement', category='rc',
                sequences=[reverse_complement(s) for s in sequences],
                params={}, description='Reverse complement',
            )
        return results


suite = DNAPerturbationSuite(seed=320, snp_rates=config['snp_rates'])

test_perturbed = suite.run_all(sequences[:3])
for name, pset in test_perturbed.items():
    dists = [
        sum(a != b for a, b in zip(o, p)) / len(o)
        for o, p in zip(sequences[:3], pset.sequences)
    ]
    print(f"  {name}: mean_hamming={np.mean(dists):.4f}")
print("Perturbation suite ready")

In [ ]:
# Caduceus Model Wrapper
#
# Native mamba-ssm only -- no pure-Python shim.
# builds mamba-ssm from source, so native CUDA kernels are available.
# This gives full pretrained weights, no missing keys, no NaN.
#
# CRITICAL: Caduceus "ph" (Parameter sHaring) models require TWO things:
#   1. Shared weights between mamba_fwd and mamba_rev  (weight tying)
#   2. config.rcps = True  (activates the geometric FLIP on inputs/outputs)
# Without #2, both branches see the same un-flipped input and the model
# is effectively unidirectional -- destroying RC equivariance entirely.

import os
import torch
import gc
from transformers import AutoModel, AutoTokenizer, AutoConfig
import transformers.modeling_utils as _mu


def _patch_tied_weights():
    """Patch transformers for Caduceus compatibility.

    Caduceus's tie_weights() doesn't define all_tied_weights_keys or accept
    the missing_keys/recompute_mapping kwargs that newer transformers expects.
    This is needed even with native mamba-ssm.
    """
    if getattr(_mu, '_caduceus_patched', False):
        return

    orig_mark = _mu.PreTrainedModel.mark_tied_weights_as_initialized

    def safe_mark(self):
        if not hasattr(self, 'all_tied_weights_keys'):
            self.all_tied_weights_keys = {}
        return orig_mark(self)

    _mu.PreTrainedModel.mark_tied_weights_as_initialized = safe_mark

    orig_finalize = _mu.PreTrainedModel._finalize_load_state_dict

    @staticmethod
    def safe_finalize(model, load_config, load_info):
        if not hasattr(model, 'all_tied_weights_keys'):
            model.all_tied_weights_keys = {}
        orig_tie = model.tie_weights
        def _patched_tie(missing_keys=None, recompute_mapping=False, **kwargs):
            return orig_tie()
        model.tie_weights = _patched_tie
        return orig_finalize(model, load_config, load_info)

    _mu.PreTrainedModel._finalize_load_state_dict = safe_finalize
    _mu._caduceus_patched = True
    print("  Patched transformers for Caduceus compatibility")

_patch_tied_weights()


def cleanup_gpu():
    """Free GPU memory between models."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  GPU memory after cleanup: {mem:.2f} GB")


def make_caduceus_embedding_fn(model_name, batch_size=4):
    """Load Caduceus model with native mamba-ssm and return embedding function."""
    print(f"Loading {model_name}...")
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"  Device: {device}")

    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

    # Load config -- respect the checkpoint's native architecture.
    # IMPORTANT: PH vs PS are fundamentally different architectures:
    #   PH ("Pre-trained Human"): bidirectional=True, rcps=False
    #       -> Two separate Mamba branches, NO geometric flip, NOT RC-equivariant
    #   PS ("Parameter Sharing"):  bidirectional=True, rcps=True
    #       -> RCPSWrapper flips input for reverse branch, weight-tied, RC-equivariant
    # DO NOT force rcps=True on PH checkpoints -- it wraps layers in RCPSWrapper
    # which adds .submodule. to all paths, causing a complete state_dict mismatch.
    cfg = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
    is_rcps = getattr(cfg, 'rcps', False)
    print(f"  Config: rcps={is_rcps}, bidirectional={getattr(cfg, 'bidirectional', 'N/A')}")
    if not is_rcps:
        print("  NOTE: This is a PH (non-RCPS) model. It will NOT show RC equivariance.")
        print("        Use caduceus-ps_* models for RC equivariant experiments.")

    # Disable fused Triton norms if they're not available
    try:
        from mamba_ssm.ops.triton.layer_norm import layer_norm_fn, rms_norm_fn
        if layer_norm_fn is None or rms_norm_fn is None:
            raise ImportError("fused norms are None")
    except (ImportError, AttributeError):
        if hasattr(cfg, 'fused_add_norm'):
            cfg.fused_add_norm = False
            print("  Disabled fused_add_norm (Triton norms not available)")
        if hasattr(cfg, 'fused_dropout_add_ln'):
            cfg.fused_dropout_add_ln = False

    model = AutoModel.from_pretrained(
        model_name, config=cfg, trust_remote_code=True,
    ).to(device).eval()

    # Re-tie mamba_rev weights to mamba_fwd after checkpoint loading.
    # from_pretrained can break weight ties by assigning new tensors.
    if hasattr(model, 'tie_weights'):
        model.tie_weights()
        print("  Re-tied mamba_rev <-> mamba_fwd weights after loading")

    # Verify weight ties (path differs by architecture)
    # PS (rcps=True): layers wrap in RCPSWrapper -> .mixer.submodule.mamba_fwd
    # PH (rcps=False): direct access            -> .mixer.mamba_fwd
    try:
        layer0_mixer = model.backbone.layers[0].mixer
        inner = layer0_mixer.submodule if hasattr(layer0_mixer, 'submodule') else layer0_mixer
        fwd_ptr = inner.mamba_fwd.in_proj.weight.data_ptr()
        rev_ptr = inner.mamba_rev.in_proj.weight.data_ptr()
        tied = fwd_ptr == rev_ptr
        print(f"  Weight tie verification: {'OK - same tensor' if tied else 'SEPARATE tensors (expected for PH, broken for PS)'}")
        if is_rcps and not tied:
            print("  WARNING: PS model but weights not tied! Forcing re-tie...")
            for layer in model.backbone.layers:
                mix = layer.mixer.submodule if hasattr(layer.mixer, 'submodule') else layer.mixer
                mix.mamba_rev.in_proj.weight = mix.mamba_fwd.in_proj.weight
                if hasattr(mix.mamba_fwd.in_proj, 'bias') and mix.mamba_fwd.in_proj.bias is not None:
                    mix.mamba_rev.in_proj.bias = mix.mamba_fwd.in_proj.bias
                mix.mamba_rev.out_proj.weight = mix.mamba_fwd.out_proj.weight
                if hasattr(mix.mamba_fwd.out_proj, 'bias') and mix.mamba_fwd.out_proj.bias is not None:
                    mix.mamba_rev.out_proj.bias = mix.mamba_fwd.out_proj.bias
            fwd2 = inner.mamba_fwd.in_proj.weight.data_ptr()
            rev2 = inner.mamba_rev.in_proj.weight.data_ptr()
            print(f"  After manual re-tie: {'OK' if fwd2 == rev2 else 'STILL BROKEN'}")
    except Exception as e:
        print(f"  Weight tie check skipped: {e}")

    n_params = sum(p.numel() for p in model.parameters()) / 1e6
    if torch.cuda.is_available():
        mem = torch.cuda.memory_allocated() / 1e9
        print(f"  Params: {n_params:.1f}M | GPU mem: {mem:.2f} GB")
    else:
        print(f"  Params: {n_params:.1f}M")

    @torch.no_grad()
    def embed(sequences):
        embeddings = []
        n_batches = (len(sequences) + batch_size - 1) // batch_size

        for i in range(0, len(sequences), batch_size):
            batch = sequences[i:i + batch_size]
            batch_idx = i // batch_size + 1

            if batch_idx % 20 == 0 or batch_idx == n_batches:
                print(f"    Batch {batch_idx}/{n_batches}", end='\r')

            tokens = tokenizer(
                batch, return_tensors='pt', padding=True,
                truncation=True, max_length=131072,
            )
            tokens = {k: v.to(device) for k, v in tokens.items()}

            outputs = model(**tokens, output_hidden_states=True)
            hidden = outputs.hidden_states[-1]

            # RCPS models have doubled hidden dim: [fwd_half; rev_half].
            # For seq s the model outputs [a; b], for rc(s) it outputs [b; a].
            # Averaging the halves gives RC-INVARIANT embeddings: (a+b)/2 == (b+a)/2.
            if is_rcps:
                d = hidden.shape[-1]
                half1 = hidden[..., :d//2]
                half2 = hidden[..., d//2:]
                hidden = (half1 + half2.flip(dims=[-1])) / 2

            if 'attention_mask' in tokens:
                mask = tokens['attention_mask'].unsqueeze(-1)
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
            else:
                pooled = hidden.mean(dim=1)
            embeddings.append(pooled.cpu().numpy())

        print()
        return np.concatenate(embeddings, axis=0)

    return embed, model, tokenizer, n_params

print("Caduceus wrapper ready (native mamba-ssm)")

In [ ]:
# ============================================================
# SANITY CHECK: Does Caduceus actually "see" RC symmetry?
# ============================================================
#
# This cell is standalone. Run it after the model wrapper cell above.
# It loads the smallest Caduceus model, embeds 50 random DNA
# sequences AND their reverse complements, and reports the
# cosine similarity between each pair.
#
# RC equivariance requires BOTH:
#   1. Shared weights (mamba_rev tied to mamba_fwd)
#   2. Geometric flipping (config.rcps=True activates the flip logic in RCPSWrapper)
#
# This only works with caduceus-PS models (rcps=True natively).
# PH models (rcps=False) will show zero RC gap -- that's expected, not a bug.
#
# Expected results (PS model with RCPS active):
#   - Centered cos(fwd, rc):  >> cos(rand, rand)  (gap > 0.3)
#   - L2 ratio (rc / rand):   < 0.5

import numpy as np
import torch
from scipy.spatial.distance import cosine as cosine_dist

COMPLEMENT = {'A': 'T', 'T': 'A', 'C': 'G', 'G': 'C'}

def rc(seq):
    """Reverse complement a DNA sequence."""
    return ''.join(COMPLEMENT.get(b, b) for b in reversed(seq))

# --- Load smallest model ---
test_model_name = config['models'][0]
print(f"Loading {test_model_name} for RC sanity check...")
embed_fn, model_obj, tok_obj, n_params = make_caduceus_embedding_fn(test_model_name, batch_size=8)

# Confirm RCPS config
is_rcps = getattr(model_obj.config, 'rcps', False)
print(f"\n  Config verification:")
print(f"    rcps:           {is_rcps}  {'<-- RC equivariance ACTIVE' if is_rcps else '<-- NO RC equivariance (PH model)'}")
print(f"    bidirectional:  {getattr(model_obj.config, 'bidirectional', 'MISSING')}")
print(f"    complement_map: {getattr(model_obj.config, 'complement_map', 'MISSING')}")
if not is_rcps:
    print("\n  WARNING: This is a PH model (rcps=False). RC test will likely FAIL.")
    print("  Switch to caduceus-ps_* models in the config cell for RC equivariance.")

# --- Generate test sequences ---
rng = np.random.default_rng(42)
N_TEST = 50
test_seqs = [''.join(rng.choice(list('ACGT'), size=200)) for _ in range(N_TEST)]
test_rc   = [rc(s) for s in test_seqs]

# --- Embed both ---
print(f"Embedding {N_TEST} sequences + their reverse complements...")
emb_fwd = embed_fn(test_seqs)
emb_rc  = embed_fn(test_rc)

# --- Compute cosine similarity (raw and centered) ---
# Raw cosine can be misleading if embeddings are anisotropic (all point
# in roughly the same direction). Centered cosine subtracts the global
# mean embedding first, revealing actual structural differences.

all_emb = np.concatenate([emb_fwd, emb_rc], axis=0)
global_mean = all_emb.mean(axis=0)
emb_fwd_c = emb_fwd - global_mean
emb_rc_c  = emb_rc  - global_mean

raw_cosines = []
centered_cosines = []
for i in range(N_TEST):
    raw_cosines.append(1.0 - cosine_dist(emb_fwd[i], emb_rc[i]))
    centered_cosines.append(1.0 - cosine_dist(emb_fwd_c[i], emb_rc_c[i]))

raw_cosines = np.array(raw_cosines)
centered_cosines = np.array(centered_cosines)

# Baseline: random unrelated pairs (centered)
rand_centered = []
for i in range(N_TEST // 2):
    j = (i + N_TEST // 2) % N_TEST  # pair unrelated sequences
    rand_centered.append(1.0 - cosine_dist(emb_fwd_c[i], emb_fwd_c[j]))
rand_centered = np.array(rand_centered)

print("\n" + "=" * 60)
print("REVERSE COMPLEMENT SYMMETRY TEST")
print("=" * 60)
print(f"  Model:         {test_model_name.split('/')[-1]}")
print(f"  Sequences:     {N_TEST} x 200bp")
print()
print(f"  --- Raw cosine (can be misleading if anisotropic) ---")
print(f"  cos(fwd, rc):      {raw_cosines.mean():.4f} +/- {raw_cosines.std():.4f}")
print()
print(f"  --- Centered cosine (removes global mean direction) ---")
print(f"  cos(fwd, rc):      {centered_cosines.mean():.4f} +/- {centered_cosines.std():.4f}")
print(f"  cos(rand, rand):   {rand_centered.mean():.4f} +/- {rand_centered.std():.4f}")
print(f"  Gap (rc - rand):   {centered_cosines.mean() - rand_centered.mean():.4f}")
print()

gap = centered_cosines.mean() - rand_centered.mean()
if gap > 0.30:
    print("  PASS -- Clear RC equivariance. The model treats seq and RC as near-identical.")
elif gap > 0.10:
    print("  PARTIAL -- Some RC awareness, but not fully equivariant.")
elif centered_cosines.mean() > 0.90:
    print("  ANISOTROPIC -- High raw similarity but due to embedding collapse, not RC awareness.")
    print("  The model maps everything to the same region. Centered test is more informative.")
else:
    print("  FAIL -- No RC equivariance detected. mamba_rev weights may be broken.")
    print("         cos(fwd,rc) is not meaningfully higher than cos(rand,rand).")
print("=" * 60)

# Euclidean distance as secondary check (not affected by anisotropy)
l2_rc   = np.array([np.linalg.norm(emb_fwd[i] - emb_rc[i]) for i in range(N_TEST)])
l2_rand = np.array([np.linalg.norm(emb_fwd[i] - emb_fwd[(i + N_TEST//2) % N_TEST]) for i in range(N_TEST//2)])
print(f"\n  L2 distance check:")
print(f"    fwd vs rc:    {l2_rc.mean():.4f} +/- {l2_rc.std():.4f}")
print(f"    rand vs rand: {l2_rand.mean():.4f} +/- {l2_rand.std():.4f}")
print(f"    Ratio:        {l2_rc.mean() / max(l2_rand.mean(), 1e-8):.4f} (< 0.5 = strong RC equivariance)")

# Cleanup
del model_obj, tok_obj
cleanup_gpu()
print("\nDone. Proceed to experiment cells.")

In [ ]:
# Evaluation Harness

from evaluation_harness import StabilityHarness

harness = StabilityHarness(
    window_size=0,
    metric='cosine',
    n_splits=30,
    seed=320,
    max_samples=2500,
    n_bootstrap=config['n_bootstrap'],
)

print(f"Harness configured")

In [ ]:
# Run Experiment

import os
import time
import pandas as pd
from evaluation_harness import ModelReport

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

print('=' * 70)
print(f'CADUCEUS + REAL GENOMIC DNA -- Phase: {PHASE.upper()}')
print('=' * 70)

reports = []
model_param_counts = []
all_detailed_rows = []

for model_idx, model_name in enumerate(config['models']):
    print(f"\n{'='*70}")
    print(f"[{model_idx+1}/{len(config['models'])}] {model_name}")
    print('=' * 70)

    start_time = time.time()

    embed_fn, model_obj, tokenizer_obj, n_params_m = make_caduceus_embedding_fn(
        model_name, config['batch_size']
    )
    model_param_counts.append(n_params_m)

    print('\nGenerating perturbations...')
    perturbed_sets = suite.run_all(sequences)

    safe_name = model_name.replace('/', '_').replace('-', '_') + '_realdna'
    cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

    if os.path.exists(cache_clean):
        print(f'Loading cached: {cache_clean}')
        embeddings_clean = np.load(cache_clean)
    else:
        print(f'Computing clean embeddings ({len(sequences)} sequences)...')
        embeddings_clean = embed_fn(sequences)
        np.save(cache_clean, embeddings_clean)
        print(f'  Cached to {cache_clean}')

    print(f'  Shape: {embeddings_clean.shape}')

    perturbed_embeddings = {}
    for pert_name, pset in perturbed_sets.items():
        cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
        if os.path.exists(cache_pert):
            print(f'  Loading cached: {pert_name}')
            perturbed_embeddings[pert_name] = np.load(cache_pert)
        else:
            print(f'  Embedding: {pert_name}...')
            perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
            np.save(cache_pert, perturbed_embeddings[pert_name])

    del model_obj, tokenizer_obj
    cleanup_gpu()

    print('\nRunning Shesha evaluation...')
    report = harness.evaluate_all_perturbations(
        model_name=model_name,
        embeddings_clean=embeddings_clean,
        perturbed_dict=perturbed_embeddings,
    )
    reports.append(report)

    # Per-perturbation rows
    short_name = model_name.split('/')[-1]
    for pert_name, metrics in report.perturbation_breakdown().items():
        all_detailed_rows.append({
            'model': short_name,
            'size_M': round(n_params_m, 1),
            'perturbation': pert_name,
            'rdm_similarity': metrics.get('rdm_similarity_score', 0),
            'rdm_drift': metrics.get('rdm_drift', 0),
            'pert_stability': metrics.get('perturbation_stability_score', 0),
            'pert_magnitude': metrics.get('perturbation_magnitude', 0),
            'composite': metrics.get('composite_stability', 0),
        })

    elapsed = time.time() - start_time
    summary = report.summary()
    print(f'\nCompleted in {elapsed/60:.1f} min')
    print(f'  Composite Stability: {summary["mean_composite_stability"]:.4f}')
    print(f'  RDM Similarity:      {summary["mean_rdm_similarity_score"]:.4f}')

# Save detailed CSV
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/caduceus_realdna_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nPer-perturbation results:')
print(df_detailed.to_string(index=False))
print(f'\nSaved to {detailed_path}')

print('\n' + '=' * 70)
print('EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Results & Visualization

from evaluation_harness import compare_models
import matplotlib.pyplot as plt

comparison = compare_models(reports, output_dir=RESULTS_DIR)

print('\n' + '=' * 70)
print('CADUCEUS + REAL DNA SUMMARY')
print('=' * 70 + '\n')

for model_name, data in comparison['models'].items():
    s = data['summary']
    short = model_name.split('/')[-1]
    print(f'{short}:')
    print(f'  Composite Stability:  {s["mean_composite_stability"]:.4f} +/- {s["std_composite_stability"]:.4f}')
    print()

# Plot using actual param counts
summaries = [r.summary() for r in reports]
values = [s['mean_composite_stability'] for s in summaries]

plt.figure(figsize=(10, 6))
plt.plot(model_param_counts, values, 'o-', linewidth=2, markersize=10, color='tab:purple')
plt.xlabel('Parameters (M)')
plt.ylabel('Composite Stability')
plt.title('Caduceus (SSM) on Real Genomic DNA: Stability vs. Size', fontweight='bold')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/caduceus_realdna_scaling_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

print(f'Saved to {RESULTS_DIR}/caduceus_realdna_scaling_{PHASE}.png')

In [ ]:
# Cross-Experiment Comparison
#
# Compare: Caduceus synthetic vs real DNA, and vs ESM-2

import matplotlib.pyplot as plt
import pandas as pd
import glob

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Panel 1: Caduceus synthetic vs real DNA ---
ax1 = axes[0]
cad_syn_files = glob.glob('results/caduceus_scaling_*_detailed.csv')
if cad_syn_files:
    df_cad_syn = pd.read_csv(cad_syn_files[0])
    syn_agg = df_cad_syn.groupby('size_M')['composite'].mean().reset_index()
    ax1.plot(syn_agg['size_M'], syn_agg['composite'], 'o--',
             color='tab:gray', linewidth=2, markersize=10, label='Caduceus (Synthetic DNA)', alpha=0.6)

real_agg = df_detailed.groupby('size_M')['composite'].mean().reset_index()
ax1.plot(real_agg['size_M'], real_agg['composite'], 's-',
         color='tab:purple', linewidth=2, markersize=10, label='Caduceus (Real DNA)')

ax1.set_xlabel('Parameters (M)', fontsize=12)
ax1.set_ylabel('Composite Stability', fontsize=12)
ax1.set_title('Caduceus: Synthetic vs Real DNA', fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Panel 2: All architectures ---
ax2 = axes[1]
esm2_files = glob.glob(f'{RESULTS_DIR}/esm2_scaling_*_detailed.csv')
if esm2_files:
    df_esm2 = pd.read_csv(esm2_files[0])
    esm2_agg = df_esm2.groupby('size_M')['composite'].mean().reset_index()
    ax2.plot(esm2_agg['size_M'], esm2_agg['composite'], 'o-',
             color='tab:red', linewidth=2, markersize=10, label='ESM-2 (Transformer)')

ax2.plot(real_agg['size_M'], real_agg['composite'], 's-',
         color='tab:blue', linewidth=2, markersize=10, label='Caduceus (SSM)')

ax2.set_xscale('log')
ax2.set_xlabel('Parameters (M)', fontsize=12)
ax2.set_ylabel('Composite Stability', fontsize=12)
ax2.set_title('Transformer vs SSM: The Geometric Tax', fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/caduceus_realdna_crossexp_{PHASE}.png', dpi=300, bbox_inches='tight')
plt.show()

if not esm2_files:
    print('Note: ESM-2 results not found. Run ESM-2 experiment and copy CSV to results/')
if not cad_syn_files:
    print('Note: Caduceus synthetic results not found. Run that experiment first.')